# Fuel versus fare: The effect of relative travel costs on modal split

Authors: Daan Devenijns, Maarten Hogendoorn, Hidde Okhuijsen, Gijs Six, Nova Swanenburg 

# 1. Introduction

The transition towards sustainable mobility requires a clear understanding of what drives travellers to choose public transport over private vehicles. While factors such as travel time, reliability, and comfort play an important role, the cost of travel remains one of the primary drivers behind commuting decisions. Over recent years, high volatility in fuel prices, combined with rising railway operational costs and fare adjustments, has continuously shifted the relative cost competitiveness between driving and taking the train.

This study investigates the impact of the ratio between fuel prices and train ticket prices on travellers' mode choice (car vs train). Rather than analysing the costs of a single mode, looking at the relative price ratio offers direct insight into the tipping point where passengers choose to change their mode of travel, without having to account for inflation.

# 2. Research Objective

The objective of this project is to quantify the impact of changes in fuel prices as well as train ticket prices on passenger mode choice, and predict future behaviour by estimating price elasticities.

#### Research question:
_What is the influence of the price ratio between fuel and train tickets on the choice between car and train?_

#### Sub questions:
* How has the ratio between fuel prices and train ticket prices evolved over the last 25 years?
* To what extent does a change in the fuel-to-fare price ratio affect the modal split between car and train?
* What is the price elasticity of modal choice with respect to the fuel-fare ratio?

#### Geografical and temporal scale:
To evaluate the relationship between transport pricing and public transport usage, this study examines the Netherlands from 1999 to 2025. This timeframe is determined by the availability of consistent, long-term longitudinal data from Statistics Netherlands (CBS) regarding mode usage and the Consumer Price Index (CPI) for both car and train transport.

# 3. Data Pipeline

To compare the relative price of rail travel to fuel costs, data from CBS will be used. Specifically, table 83131NED (*Consumentenprijzen; prijsindex 2015=100, 1996–2025*) is selected. This table provides the Consumer Price Index (CPI) across various expenditure categories, including passenger transport by train and different fuel types. 

To determine the modal split, CBS table 84755NED (*Mobiliteitstrend; per rit, vervoerwijzen, reismotief, leeftijd en geslacht*) will be used. This dataset covers the period from 1999 to 2025 and contains the average number of trips made per person per transport mode, alongside the average trip distance.

This section outlines the definition and function of the Consumer Price Index (CPI), followed by a detailed description of the data cleaning and processing pipeline implemented in Python.

## 3.1 Consumer Price Index (CPI)
The Consumer Price Index (CPI) measures the average price change of a representative collection of goods and services consumed by households. It is used as a tool to follow yearly inflation and consumer purchasing power. If the CPI rises from 100 in one year to 110 the next year, the representative collection has become 10 percent more expensive.

The CBS data divides the CPI into categories such as education, restaurants and hotels, or transport, and into countless subcategories for all kinds of things such as bread, software, or jewellery. The values of these subcategories can be used to assess the relative costs of products or services in that category. This study uses the CPI of the subcategories passenger transport by train (subcategory 073110), diesel (072210), petrol (072220), and LPG (072230) to evaluate change in time of the relative costs of passenger train transport compared to the costs of different fuel types.  Because both price series are normalised to the same base year (the total CPI in 2015 is set to 100), comparing the rail CPI directly to a fuel CPI filters out general inflation.

## 3.2 Data Processing Pipeline
This section explains how the raw data from CBS was processed into usable data using Python and the pandas package. The code can be found in the GitHub repository in de folder code > data_pipeline.ipynb. The resulting processed data in the folder data > processed_data.

### 3.2.1 Processing CPI Data
The function `process_cpi_data` imports and cleans the raw CPI tables, and combines the CPI data from passenger transport by train and the different fuel types into one dataframe. The code follows these steps:
- For each CPI data file:
  - Import the data file and store it in a pandas dataframe.
    - Skip the first 4 rows because these only contain metadata.
    - Delete the last row. This row only contains metadata.
  - Translate the months to English and convert to datetime format. This will allow for easier plooting later.
  - Replace all ',' decimals with '.' and convert to floats to make sure all values are valid numbers.
- Combine all dataframes into one dataframe.
- Export the new dataframe to a csv file using the export_dataframe function.

before and after

### 3.2.2 Processing Modal Split and Travel Behaviour Data
The `process_mode_data` function imports and cleans the modal split dataset:
- Import the data file and store it as a pandas dataframe.
  - Skip the first 4 rows because these only contain metadata.
  - Set the first 3 rows as column indices.
  - Set the first 2 columns as row indices.
  - Delete the last row. This row only contains metadata.
- Delete the asterix after the years and convert the data to datetime format.
- Multi-index cleaning: Semicolon-delimited files containing a three-level column hierarchy and a two-level index ('Date' and 'Travel purpose') are imported. Asterisks marking provisional yearly estimates are stripped before converting dates into annual datetime objects.
- Translation and dictionary mapping: Dutch travel purposes (e.g., 'Van en naar het werk', 'Vrije tijd') and modes (e.g., 'Auto (bestuurder)', 'Trein') are translated into British English to maintain international academic standards.
- Header consolidation: The hierarchical column levels (mode, indicator, and unit) are consolidated from three levels into a two-level MultiIndex, explicitly incorporating the physical units (trips, km, min) directly into the indicator labels.
- Aggregation of total car mobility: Travel survey data distinguishes between car drivers and car passengers. To evaluate private road transport as a whole, a combined 'Car (total)' mode is constructed:
  - Daily car trips are aggregated by summing driver and passenger trips.
  - Average trip distance and trip travel duration for total car travel are calculated as trip-weighted averages rather than simple arithmetic means, ensuring mathematical consistency.
- Metric derivation: Average daily distance travelled (km/person/day) and daily travel time (min/person/day) are derived for each mode by multiplying daily trip frequencies by the respective average trip metric.
- Column ordering: Columns are sorted alphabetically by mode for structured visual inspection.
Multiplying these metrics yields the average daily kilometres travelled per person for each transport mode. Missing years within the time series will have to be interpolated.

before and after

# References

Centraal Bureau voor de Statistiek. (2024). *Consumentenprijzen; prijsindex 2015=100* (Dataset 83131NED) [Data set]. StatLine. https://opendata.cbs.nl/#/CBS/nl/dataset/83131NED/table

Centraal Bureau voor de Statistiek. (2024). *Reizigersvervoer per spoor; reizigerskilometers* (Dataset 84755NED) [Data set]. StatLine. https://opendata.cbs.nl/#/CBS/nl/dataset/84755NED/table

Compendium voor de Leefomgeving. (2026). *Aantal motorvoertuigen actief, 2019-2026* (Indicator 0026, versie 28) [Data set]. CBS, PBL, RIVM, WUR. https://www.clo.nl/indicatoren/nl002628-aantal-motorvoertuigen-actief-2019-2026

# Contribution Statement

All authors contributed to the coding of this project. Below is a breakdown of each author's primary focus areas.

**Daan Devenijns:**
- Background research (literature on fuel price volatility and modal choice)
- Data collection (fuel prices)
- Coding

**Maarten Hogendoorn:**
- Data collection (train ticket prices / fare data)
- Data modelling (price elasticity estimation)
- Coding

**Hidde Okhuijsen:**
- Conceptualisation (research question and sub-questions)
- Data analysis
- Coding

**Gijs Six:**
- Data pipeline (cleaning and merging data sources)
- Visualisation
- Coding

**Nova Swanenburg:**
- Background research (modal split literature)
- Visualisation
- Coding

*Note: task division is preliminary and will be refined as the project progresses.*